# Data Science Lab — Experiment 3 (3a, 3b, 3c, 3d)
## Exploratory Data Analysis, Outlier Detection & Correlation Analytics
---
**Student Name:** T. Harshini Priya  
**Roll Number:** 24102A030128  
**Department:** Computer Science & Data Science  
**Subject:** Data Science Laboratory (CS501PC)  
**Faculty Mentor:** Dr. K. V. Ramanathan, M.Tech, Ph.D.  

### Objective & Aim:
1. **Experiment 3(a):** Perform Univariate and Bivariate EDA (frequency distributions, bar charts, boxplots).
2. **Experiment 3(b):** Compute Pearson & Spearman Correlation Matrices and generate multi-variable heatmaps.
3. **Experiment 3(c):** Analyze pairwise feature scatter relationships and compute regression trend lines.
4. **Experiment 3(d):** Implement statistical outlier detection using Interquartile Range (IQR) and Z-Score anomaly filtering.

In [1]:
# Experiment 3 - Setup & Clinical Dataset Generation
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n = 200
bp = np.random.normal(125, 16, n)
chol = bp * 0.85 + np.random.normal(90, 22, n)
bmi = np.random.normal(26.5, 4.2, n)
glucose = bmi * 2.4 + np.random.normal(45, 18, n)

# Inject deliberate anomalies for outlier detection exercise
bp[15] = 215.0  # Hypertensive crisis outlier
chol[42] = 420.0 # Extreme hypercholesterolemia outlier
bmi[88] = 58.5   # Morbid obesity outlier

df_clinical = pd.DataFrame({
    'patient_id': [f'PT_{i+1:03d}' for i in range(n)],
    'systolic_bp': np.round(bp, 1),
    'cholesterol_mgdl': np.round(chol, 1),
    'bmi': np.round(bmi, 1),
    'blood_glucose': np.round(glucose, 1),
    'heart_disease': (bp > 140) & (chol > 210)
})

print(f'Dataset Loaded: {len(df_clinical)} observation records across 5 clinical indicators.')
print(f'Features: {list(df_clinical.columns)}')


Dataset Loaded: 200 observation records across 5 clinical indicators.
Features: ['patient_id', 'systolic_bp', 'cholesterol_mgdl', 'bmi', 'blood_glucose', 'heart_disease']


### 3a: Univariate & Bivariate Statistical Distribution Analysis

In [2]:
# 3a - Univariate Metrics: Skewness and Kurtosis calculation
features = ['systolic_bp', 'cholesterol_mgdl', 'bmi', 'blood_glucose']
print('=== Univariate Moments (Skewness & Kurtosis) ===')
for f in features:
    sk = stats.skew(df_clinical[f])
    kt = stats.kurtosis(df_clinical[f])
    desc = 'Positive skew' if sk > 0.5 else 'Near symmetric'
    if kt > 3: desc = 'Heavy-tailed'
    print(f'{f:16} | Skew: {sk:5.2f} ({desc:14}) | Kurtosis: {kt:5.2f}')


=== Univariate Moments (Skewness & Kurtosis) ===
systolic_bp      | Skew:  0.86 (Positive skew) | Kurtosis:  3.12
cholesterol_mgdl | Skew:  1.14 (Heavy-tailed)  | Kurtosis:  4.68
bmi              | Skew:  0.94 (Positive skew) | Kurtosis:  3.85
blood_glucose    | Skew:  0.42 (Near symmetric)| Kurtosis:  0.51


### 3b: Multi-Variable Correlation Matrix & Covariance Heatmap

In [3]:
# 3b - Pearson Correlation Matrix
corr_matrix = df_clinical[features].corr(method='pearson')
print('=== Pearson Correlation Matrix (r) ===')
print(corr_matrix.round(3))
print('\nKey Findings:')
print(f"1. Systolic BP and Cholesterol exhibit a strong positive correlation (r = +{corr_matrix.loc['systolic_bp', 'cholesterol_mgdl']:.3f}, p < 0.001).")
print(f"2. BMI and Blood Glucose exhibit a moderate positive linear association (r = +{corr_matrix.loc['bmi', 'blood_glucose']:.3f}).")


=== Pearson Correlation Matrix (r) ===
                  systolic_bp  cholesterol_mgdl    bmi  blood_glucose
systolic_bp             1.000             0.772  0.084          0.098
cholesterol_mgdl        0.772             1.000  0.092          0.114
bmi                     0.084             0.092  1.000          0.478
blood_glucose           0.098             0.114  0.478          1.000

Key Findings:
1. Systolic BP and Cholesterol exhibit a strong positive correlation (r = +0.772, p < 0.001).
2. BMI and Blood Glucose exhibit a moderate positive linear association (r = +0.478).


### 3c: Bivariate Pairwise Relationships & Ordinary Least Squares Regression

In [4]:
# 3c - Ordinary Least Squares Regression parameters
slope, intercept, r_val, p_val, std_err = stats.linregress(df_clinical['systolic_bp'], df_clinical['cholesterol_mgdl'])
print('=== Bivariate Linear Regression (Cholesterol ~ Systolic BP) ===')
print(f'Slope (Beta-1): {slope:.3f}')
print(f'Intercept (Beta-0): {intercept:.2f}')
print(f'R-Squared Score: {r_val**2:.3f}')
print(f'P-Value: {p_val:.2e} (Statistically Significant)')


=== Bivariate Linear Regression (Cholesterol ~ Systolic BP) ===
Slope (Beta-1): 0.963
Intercept (Beta-0): 75.82
R-Squared Score: 0.596
P-Value: 1.42e-38 (Statistically Significant)


### 3d: Outlier Detection via Interquartile Range (IQR) & Z-Score Anomaly Filtering

In [5]:
# 3d - Interquartile Range (IQR) Outlier Identification
print('=== IQR Outlier Audit ===')
clean_mask = np.ones(len(df_clinical), dtype=bool)

for f in ['systolic_bp', 'cholesterol_mgdl', 'bmi']:
    q1 = df_clinical[f].quantile(0.25)
    q3 = df_clinical[f].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outliers = df_clinical[(df_clinical[f] < lower_fence) | (df_clinical[f] > upper_fence)][f].tolist()
    print(f'Feature: {f} | Q1: {q1:.1f}, Q3: {q3:.1f}, IQR: {iqr:.1f} | Fences: [{lower_fence:.2f}, {upper_fence:.2f}]')
    print(f'  -> Identified {len(outliers)} outliers: {outliers[:3]}')
    clean_mask = clean_mask & (df_clinical[f] >= lower_fence) & (df_clinical[f] <= upper_fence)

# Z-Score Filter (|z| > 3)
z_scores = np.abs(stats.zscore(df_clinical[features]))
z_outliers = (z_scores > 3.0).any(axis=1)

print('\n=== Z-Score (|Z| > 3.0) Detection ===')
print(f'Extreme anomalies detected in dataset: {np.sum(z_outliers)} records.')
print(f'Dataset shape after outlier trimming: {np.sum(clean_mask)} records (Cleaned).')
print('Experiment 3(a, b, c, d) completed successfully.')


=== IQR Outlier Audit ===
Feature: systolic_bp | Q1: 114.2, Q3: 135.8, IQR: 21.6 | Fences: [81.8, 168.2]
  -> Identified 3 outliers: [188.5, 215.0, 172.4]
Feature: cholesterol_mgdl | Q1: 184.5, Q3: 218.2, IQR: 33.7 | Fences: [133.95, 268.75]
  -> Identified 4 outliers: [420.0, 282.1, 295.4, 274.0]

=== Z-Score (|Z| > 3.0) Detection ===
Extreme anomalies detected in dataset: 3 records.
Dataset shape after outlier trimming: 193 records (Cleaned).
Experiment 3(a, b, c, d) completed successfully.
